In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage, SystemMessage

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [3]:
config = {"configurable": {"thread_id":"test-1"}}

In [4]:
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]
for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"messages : {response}")
    print(f"messages : {len(response['messages'])}")

messages : {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='5d1c99bc-5fb3-495d-b567-b13aaaa0a811'), AIMessage(content="<think>\nOkay, so the question is asking what is 2 plus 2. Hmm, that seems pretty straightforward. Let me think. I remember from basic math that addition is combining two numbers. So 2 plus 2 would be adding two and another two together. Let me count them out. If I have two apples and then get two more apples, how many do I have total? One, two, three, four. So that's four apples. Wait, is there any other way this could be interpreted? Maybe in some contexts, like in different number bases? For example, in base 3, 2 plus 2 would be different. Let's see. In base 3, the digits go 0, 1, 2. So adding 2 and 2 in base 3 would be 11, which is 4 in decimal. But the question doesn't specify a base, so I think it's safe to assume it's base 10. Another thought: maybe in some special mathematical context, like modulo arithmetic? If

In [5]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city:str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    tools=[search_hotels],
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("tokens", 550),
            keep=("tokens",200)
        )
    ]
)

config = {"configurable": {'thread_id': "test-1"}}

def count_tokens(messages):
    totla_chars = sum(len(str(m.content)) for m in messages)
    return totla_chars // 4

In [6]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"find the hotel for {city}")]},
        config=config
    )

tokens = count_tokens(response['messages'])
print(f"{city} ~ {tokens} tokens, {len(response['messages'])} messages")
print(f"{response['messages']}")

Singapore ~ 908 tokens, 6 messages
[HumanMessage(content='Here is a summary of the conversation to date:\n\n<think>\nOkay, let me try to work through this. The user wants me to extract the most relevant context from their conversation history to replace it, saving space. They provided a structure with four sections: SESSION INTENT, SUMMARY, ARTIFACTS, and NEXT STEPS.\n\nFirst, looking at the messages, the user has been switching cities for hotel searches: London, Tokyo, New York, now Dubai. The latest message is about Dubai. The assistant provided a hotel list for Dubai with three options. The user\'s intent seems to be finding hotels in Dubai, similar to previous searches.\n\nFor SESSION INTENT, the main goal is the Dubai hotel search. The user hasn\'t specified preferences like budget or star ratings yet, so I need to note that. \n\nIn the SUMMARY section, I should mention the transition from previous cities to Dubai, the lack of filters, and the structured format used before (like t

In [8]:
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

@tool
def search_hotels(city:str)->str:
    """find hotels in {city}"""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("fraction", 0.005),
            keep=("fraction", 0.002)
        )
    ]
)

config = {"configurable": {"thread_id":"test-1"}}

def token_counter(messages):
    return sum(len(str(m.content)) for m in messages) // 4

cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"find hotels in {city}")]},
        config=config
    )

    tokens = token_counter(response['messages'])
    fraction = tokens / 131072 # model context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~73 tokens (0.0557%), 4 msgs
[HumanMessage(content='find hotels in Paris', additional_kwargs={}, response_metadata={}, id='85c41124-77a9-46c4-acc5-864d960aa763'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking to find hotels in Paris. Let me check the available tools. There\'s a function called search_hotels that takes a city parameter. The required parameter is city, and it\'s a string. Since the user specified Paris, I need to call this function with "Paris" as the city argument. I\'ll make sure the JSON is correctly formatted with the city name in quotes. No other parameters are needed here. Alright, time to structure the tool call properly.\n', 'tool_calls': [{'id': 'fmeks69wm', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 124, 'prompt_tokens': 150, 'total_tokens': 274, 'completion_time': 0.181248359, 'completion_tokens_details': 

In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool
from langchain.messages import  HumanMessage


def read_email_tool(email_id:str)->str:
    """Mock function for reading emails by its ID"""
    return f"content of email for ID: {email_id}"
    

def sent_email_tool(receipent:str, subject:str, body:str)->str:
    """Mock function for sending an email"""
    return f"email sent to {receipent} with the subject '{subject}'"

In [12]:
agent= create_agent(
    model= "groq:qwen/qwen3-32b",
    tools=[read_email_tool, sent_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "sent_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

In [13]:
config = {"configurable" : {"thread_id" : "test-approve"}}
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config
)

In [14]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='55b81e07-7ba7-4fa3-85e3-b685213371f5'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to john@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's the sent_email_tool which requires recipient, subject, and body. I need to make sure all required parameters are included. The recipient is provided as john@test.com, subject is 'Hello', and body is 'How are you?'. All required fields are present. I'll call the sent_email_tool with these parameters.\n", 'tool_calls': [{'id': 'k8yr1dmtr', 'function': {'arguments': '{"body":"How are you?","receipent":"john@test.com","subject":"Hello"}', 'name': 'sent_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 254, 'total_tokens': 

In [15]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"approve"}
                ]
            }
        ),
        config=config
    )

    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email has been successfully sent to john@test.com with the subject "Hello" and the body "How are you?". Let me know if you need any further assistance!


In [24]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool
from langchain.messages import  HumanMessage


def read_email_tool(email_id:str)->str:
    """Mock function for reading emails by its ID"""
    return f"content of email for ID: {email_id}"
    

def sent_email_tool(receipent:str, subject:str, body:str)->str:
    """Mock function for sending an email"""
    return f"email sent to {receipent} with the subject '{subject}'"

agent= create_agent(
    model= "groq:qwen/qwen3-32b",
    tools=[read_email_tool, sent_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "sent_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

config = {"configurable" : {"thread_id" : "test-reject"}}
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config
)

if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"reject"}
                ]
            }
        ),
        config=config
    )

    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email sending has been canceled. Let me know if you'd like to make changes or need any other assistance!


In [25]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool
from langchain.messages import  HumanMessage


def read_email_tool(email_id:str)->str:
    """Mock function for reading emails by its ID"""
    return f"content of email for ID: {email_id}"
    

def sent_email_tool(receipent:str, subject:str, body:str)->str:
    """Mock function for sending an email"""
    return f"email sent to {receipent} with the subject '{subject}'"

agent= create_agent(
    model= "groq:qwen/qwen3-32b",
    tools=[read_email_tool, sent_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "sent_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

config = {"configurable" : {"thread_id" : "test-edit"}}
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config
)

if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")

    result = agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"edit",
                    "edited_action": {
                        "name": "sent_email_tool",      # Tool name
                        "args": {                   # New arguments
                            "recipient": "correct@email.com",
                            "subject": "Corrected Subject",
                            "body": "This was edited by human before sending"
                            }}}
                ]
            }
        ),
        config=config
    )

    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✅ Result: 
